# Data Cleaning and Saving

This notebook creates the interim datasets consumed by the downstream analysis. All datasets are produced through `DataCleaner.get_clean_data`.

## Workflow

1. Load the raw activity and vehicle data.
2. Define the shared segmentation and churn parameters.
3. Create raw datasets for exploratory analysis.
4. Create filtered, model-ready datasets for survival analysis.

## Outputs

- **All users — raw:** merged data with no user-type segmentation, cutoff filtering, or inactivity filtering.
- **Personal users — raw:** merged and segmented personal-use data with no cutoff or inactivity filtering.
- **Professional users — raw:** merged and segmented professional-use data with no cutoff or inactivity filtering.
- **Personal users — filtered:** cutoff-filtered data with incomplete vehicle metadata removed and activity truncated at the first churn event.
- **Professional users — filtered:** the equivalent model-ready dataset using the professional churn threshold.

The raw datasets support distributional exploration in notebook `01`. The filtered datasets include `churn_adjusted_date` and support interval construction and survival modelling.

## Setup and shared configuration

In [1]:
import os
import pandas as pd
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "Coding")
    if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()
)
os.chdir(PROJECT_ROOT)
PROJECT_ROOT

'c:\\Users\\Tomas\\Desktop\\Thesis Stuff\\Survival_Analysis_Thesis\\Coding'

### Data imports and cleaning parameters


In [2]:
from src.constants import paths_to_files_and_folders as const
from src.data_cleaning import DataCleaner
from src.constants.segments import PERSONAL, PROFESSIONAL
from src.constants.cleaning import DEFAULT_HHI_THRESHOLD, DEFAULT_CAR_SHARE_ABS, DEFAULT_CAR_SHARE_FRACTION

In [3]:
activity_df = pd.read_csv(const.PATH_TO_RAW_ACTIVITY_DATA_1000)
vehicle_df  = pd.read_csv(const.PATH_TO_RAW_VEHICLE_DATA_1000)
data_cleaner = DataCleaner(activity_df, vehicle_df)

# Decided churn threshold (see interval / gap analysis in notebook 01)
CHURN_THRESHOLD_DAYS_PERSONAL = PERSONAL.churn_threshold_days
CHURN_THRESHOLD_DAYS_PROFESSIONAL = PROFESSIONAL.churn_threshold_days

# Shared split criteria
HHI_THRESHOLD       = DEFAULT_HHI_THRESHOLD
CAR_SHARE_ABS       = DEFAULT_CAR_SHARE_ABS
CAR_SHARE_FRACTION  = DEFAULT_CAR_SHARE_FRACTION



## 1. Raw datasets

These datasets retain the cleaned activity history without cutoff-date or inactivity filtering. They are intended for exploratory analysis.

### 1.1 All users

In [4]:
merged_all_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=False,
    # return_personal_use_users=True,          
    filter_early_churners=False,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "merged_all_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626


Cleaning complete
Final rows: 560583
Final unique users: 2755

_Step 3_
Saving File to C:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\Data\interim\merged_all_users_raw.csv


### 1.2 Personal users


In [5]:
personal_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=True,          # personal
    filter_early_churners=False,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "personal_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Rows before user type filtering: 560583
Filtering by personal users!
Rows after user type filtering: 177113


Cleaning complete
Final rows: 177113
Final unique users: 2540

_Step 4_
Saving File to C:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\Data\interim\personal_users_raw.csv


### 1.3 Professional users


In [6]:
professional_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=False,         # professional
    filter_early_churners=True,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "professional_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Rows before user type filtering: 560583
Filtering by professional users!
Rows after user type filtering: 64062

WARNING! (Filter Early Churners)
filter_early_churners ONLY correctly functions when filter_inactivity=True
Skipping step...

Cleaning complete
Final rows: 64062
Final unique users: 215

_Step 4_
Saving File to C:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\Data\interim\professional_users_raw.csv


## 2. Filtered datasets

These model-ready datasets add cutoff-date filtering, remove incomplete vehicle metadata, and truncate each user's activity at the first churn event.

### 2.1 Personal users


In [7]:
personal_filtered = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_inactivity=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=True,          # personal
    filter_early_churners=True,
    filter_by_set_cutoff_date=True,
    transform_vehicle_end_year_to_present=True,
    filter_nan_vehicle_metadata=True,
    threshold_value=CHURN_THRESHOLD_DAYS_PERSONAL,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "personal_users_filtered.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Rows before set cutoff date filtering: 560583
Rows after set cutoff date filtering: 559272
Rows removed: 1311

_Step 4_
Rows before user type filtering: 559272
Filtering by personal users!
Rows after user type filtering: 176476

_Step 5_
Rows before vehicle metadata filtering: 176476
Rows after vehicle metadata filtering: 166240
Rows removed: 10236

_Step 6_
Filtering activity after inactivity threshold: 160 days
Rows before inactivity filtering: 166240
Rows after inactivity filtering: 132932
Rows removed: 33308
Unique users before: 2495
Unique users after: 24

### 2.2 Professional users


In [8]:
professional_filtered = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_inactivity=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=False,         # professional
    filter_early_churners=True,
    filter_by_set_cutoff_date=True,
    transform_vehicle_end_year_to_present=True,
    filter_nan_vehicle_metadata=True,
    threshold_value=CHURN_THRESHOLD_DAYS_PROFESSIONAL, 
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "professional_users_filtered.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Rows before set cutoff date filtering: 560583
Rows after set cutoff date filtering: 560468
Rows removed: 115

_Step 4_
Rows before user type filtering: 560468
Filtering by professional users!
Rows after user type filtering: 64062

_Step 5_
Rows before vehicle metadata filtering: 64062
Rows after vehicle metadata filtering: 61148
Rows removed: 2914

_Step 6_
Filtering activity after inactivity threshold: 80 days
Rows before inactivity filtering: 61148
Rows after inactivity filtering: 36477
Rows removed: 24671
Unique users before: 215
Unique users after: 215
Chu